<a href="https://colab.research.google.com/github/tristansalazar/MachineLearning-/blob/main/Unidad3/LendingClub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from IPython.display import Markdown, display

display(Markdown("""
# Machine Learning y Deep Learning

**Unidad 3**

**Práctica 2:** Regresión Logística

**Facilitador:** *Dr. José Gabriel Rodríguez Rivas*

**Alumno:** Diego Tristán Salazar Rueda

## Regresión logística

- La regresión logística toma un conjunto de variables de entrada, las características, y estima un valor objetivo.
- Es similar a la regresión lineal; la diferencia es que **calcula una suma ponderada de las características xi y el término de interceptación b, pero ejecuta este resultado a través de una función no lineal f**, para producir la salida **yhat** (predicción).
- El efecto de aplicar la función logística es comprimir la salida de la función lineal para que se limite a un rango entre 0 y 1.
- La salida puede interpretarse como **la probabilidad de que la instancia pertenezca a la clase positiva, dadas sus características de entrada**.

## Contexto de la práctica

En la primera parte se busca **comprender los objetivos y requisitos del dataset desde una perspectiva comercial**, comenzando con cómo funciona el **club de préstamos Lending Club**.

### Club de préstamos

LendingClub es una compañía estadounidense de **préstamos entre pares**, con sede en San Francisco, California. Es la plataforma de préstamos peer-to-peer más grande del mundo.

- Permite a los **prestatarios solicitar préstamos personales no garantizados** entre 1,000 y 40,000.
- Los **inversionistas pueden explorar las listas de préstamos y seleccionar en cuáles invertir según la información del prestatario, el monto, el grado y el propósito del préstamo**, con una inversión mínima de $25.
- Los inversionistas ganan dinero con los intereses.
- LendingClub gana dinero cobrando a los prestatarios una tarifa de originación y a los inversionistas una tarifa de servicio.
"""))


# Machine Learning y Deep Learning

**Unidad 3**

**Práctica 2:** Regresión Logística

**Facilitador:** *Dr. José Gabriel Rodríguez Rivas*

**Alumno:** Diego Tristán Salazar Rueda

## Regresión logística

- La regresión logística toma un conjunto de variables de entrada, las características, y estima un valor objetivo.
- Es similar a la regresión lineal; la diferencia es que **calcula una suma ponderada de las características xi y el término de interceptación b, pero ejecuta este resultado a través de una función no lineal f**, para producir la salida **yhat** (predicción).
- El efecto de aplicar la función logística es comprimir la salida de la función lineal para que se limite a un rango entre 0 y 1.
- La salida puede interpretarse como **la probabilidad de que la instancia pertenezca a la clase positiva, dadas sus características de entrada**.

## Contexto de la práctica

En la primera parte se busca **comprender los objetivos y requisitos del dataset desde una perspectiva comercial**, comenzando con cómo funciona el **club de préstamos Lending Club**.

### Club de préstamos

LendingClub es una compañía estadounidense de **préstamos entre pares**, con sede en San Francisco, California. Es la plataforma de préstamos peer-to-peer más grande del mundo.

- Permite a los **prestatarios solicitar préstamos personales no garantizados** entre 1,000 y 40,000.
- Los **inversionistas pueden explorar las listas de préstamos y seleccionar en cuáles invertir según la información del prestatario, el monto, el grado y el propósito del préstamo**, con una inversión mínima de $25.
- Los inversionistas ganan dinero con los intereses.
- LendingClub gana dinero cobrando a los prestatarios una tarifa de originación y a los inversionistas una tarifa de servicio.


In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

In [3]:
prestamos_df = pd.read_csv('lending_club_2007_2011_6_states.csv', low_memory=False)
prestamos_df.head()

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,hardship_flag,disbursement_method,debt_settlement_flag,debt_settlement_flag_date
0,2400,2400,2400.0,36 months,15.96,84.33,C,C5,NaN,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
1,10000,10000,10000.0,36 months,13.49,339.31,C,C1,AIR RESOURCES BOARD,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
2,3000,3000,3000.0,36 months,18.64,109.43,E,E1,MKC Accounting,9 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
3,5600,5600,5600.0,60 months,21.28,152.39,F,F2,NaN,4 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
4,5375,5375,5350.0,60 months,12.69,121.45,B,B5,Starbucks,< 1 year,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN


## Preprocesamiento de datos
El dataset original no incluye la variable objetivo ni los códigos numéricos que usa el modelo, por lo que se crean:
- **repaid:** 1 si el préstamo fue pagado (Fully Paid), 0 si fue cancelado (Charged Off).
- **grade_code, purpose_code, addr_state_code, home_ownership_code:** conversión de las columnas de texto a códigos numéricos.
- Los valores vacíos de **revol_util** y **pub_rec_bankruptcies** se rellenan con la media.

In [4]:
prestamos_df['repaid'] = (prestamos_df['loan_status'] == 'Fully Paid').astype(int)

prestamos_df['grade_code'] = prestamos_df['grade'].astype('category').cat.codes
prestamos_df['purpose_code'] = prestamos_df['purpose'].astype('category').cat.codes
prestamos_df['addr_state_code'] = prestamos_df['addr_state'].astype('category').cat.codes
prestamos_df['home_ownership_code'] = prestamos_df['home_ownership'].astype('category').cat.codes

prestamos_df['revol_util'] = prestamos_df['revol_util'].fillna(prestamos_df['revol_util'].mean())
prestamos_df['pub_rec_bankruptcies'] = prestamos_df['pub_rec_bankruptcies'].fillna(prestamos_df['pub_rec_bankruptcies'].mean())

prestamos_df['repaid'].value_counts()

,count
repaid,
1,16965
0,2943


## Selección de Variables predictoras
Para la selección de características la mejor opción es comprender los datos y las reglas del negocio. Se seleccionan:
- **funded_amnt:** monto financiado.
- **int_rate:** tasa de interés.
- **grade_code:** código de la calificación del crédito.
- **purpose_code:** código del propósito.
- **addr_state_code:** código del estado.
- **home_ownership_code:** código de situación de vivienda.
- **annual_inc:** ingresos anuales.
- **dti:** relación entre pagos de deuda mensuales e ingresos.
- **revol_util:** crédito utilizado en relación con el crédito total.
- **pub_rec_bankruptcies:** número de quiebras en registros públicos.

**Variable dependiente:** repaid.

**Otros aspectos a considerar:**
- Solo se usan características disponibles **antes** de que inicie el préstamo.
- Variables como recoveries o total_rec_prncp solo existen después de cerrar el préstamo y no deben incluirse.
- Si la precisión es cercana al 100%, probablemente se incluyeron variables posteriores al cierre del préstamo.

In [5]:
X = prestamos_df[['funded_amnt', "int_rate", "grade_code", 'purpose_code', 'addr_state_code',
                  'home_ownership_code', 'annual_inc', 'dti', 'revol_util',
                  'pub_rec_bankruptcies']]

y = prestamos_df["repaid"]

## Dividir dataset en conjunto de entrenamiento y prueba
60% para entrenamiento y 40% para prueba.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((11944, 10), (7964, 10), (11944,), (7964,))

## Regresión logística sin balanceo de clases
Construir un modelo de regresión logística con el atributo class_weight predeterminado. Entrenar y evaluar el modelo.

In [7]:
clf1 = LogisticRegression(random_state=0)

clf1.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(random_state=0)

In [8]:
print("Precision del clasificador en fase de entrenamiento",
      clf1.score(X_train, y_train) )

Precision del clasificador en fase de entrenamiento 0.8551574012056262


## Realizar la fase de prueba con los datos de prueba y evaluación del modelo

In [9]:
y_pred = clf1.predict(X_test)

print("\nReporte de métricas del clasificador: \n",
      classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))

print( "Precisión:", clf1.score(X_test, y_test) )


Reporte de métricas del clasificador: 
               precision    recall  f1-score   support

   No Pagado       0.00      0.00      0.00      1220
      Pagado       0.85      1.00      0.92      6744

    accuracy                           0.85      7964
   macro avg       0.42      0.50      0.46      7964
weighted avg       0.72      0.85      0.78      7964

Precisión: 0.8463083877448518


### Interpretación del reporte de Clasificación

| Clase | Precision | Recall | F1-score | Support |
|---|---|---|---|---|
| No Pagado | 0.00 | 0.00 | 0.00 | 1220 |
| Pagado | 0.85 | 1.00 | 0.92 | 6744 |

A primera vista la precisión parece alta (85%), pero no refleja el verdadero desempeño porque las clases están desequilibradas: 6,744 préstamos pagados contra 1,220 no pagados.
- El recall de "Pagado" = 1.00: predice casi todos los pagados correctamente.
- El recall de "No Pagado" = 0.00: no detecta ningún préstamo impago.
- El accuracy de 0.85 se debe a que el 85% de los datos pertenecen a la clase mayoritaria.

En otras palabras, el modelo aprendió que si siempre dice "Pagado" acierta el 85% de las veces, pero no sirve para detectar a los que no pagan, que son justo los más importantes.

## Matriz de confusión
- La diagonal principal contiene las predicciones correctas.
- La diagonal secundaria refleja los errores: falsos positivos y falsos negativos.

|  | Negativo (N) | Positivo (P) |
|---|---|---|
| **Negativo** | TN | FP |
| **Positivo** | FN | TP |

In [10]:
matriz_sin_balanceo = confusion_matrix(y_test, y_pred )
print(f'Matriz Confusion:\n', matriz_sin_balanceo)

Matriz Confusion:
 [[   0 1220]
 [   4 6740]]


### Observaciones de resultados de la matriz de confusión

|  | Predijo: No Pagado | Predijo: Pagado |
|---|---|---|
| **Real: No Pagado** | 0 (Verdaderos Negativos) | 1220 (Falsos Positivos) |
| **Real: Pagado** | 4 (Falsos Negativos) | 6740 (Verdaderos Positivos) |

- **Clase Pagado:** 6,740 créditos pagados fueron clasificados correctamente.
- **Clase No Pagado:** de 1,220 créditos impagos, el modelo no identificó ninguno. Todos los clasificó como "Pagado".

**Significado en el contexto financiero**

| Decisión del modelo | Realidad | Consecuencia |
|---|---|---|
| Predice "Pagado" y sí paga | Correcto | Se gana dinero |
| Predice "Pagado" pero no paga | Error grave | Pérdida económica importante |
| Predice "No Pagado" y no paga | Correcto | Buen filtro de riesgo |
| Predice "No Pagado" pero sí paga | Error menor | Se pierde oportunidad de préstamo |

**Posibles remedios:**
- Ajustar pesos de clase.
- Rebalancear el dataset con SMOTE o undersampling.

## Regresión logística con peso de clases balanceado
- El conjunto de datos está desbalanceado: el 85% son préstamos pagados y el 15% cancelados.
- Se puede indicar al clasificador que use un peso de clases equilibrado con **class_weight='balanced'**.
- Se usan las mismas características que en el primer modelo.

In [11]:
clf2 = LogisticRegression(random_state=0, class_weight="balanced")

clf2.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(class_weight='balanced', random_state=0)

In [12]:
print("Exactitud del modelo de entrenamiento:", clf2.score(X_train, y_train))

Exactitud del modelo de entrenamiento: 0.5882451440053583


In [13]:
y_pred2 = clf2.predict(X_test)

print("\nReporte de métricas del clasificador: \n",
      classification_report(y_test, y_pred2, target_names=["No Pagado", "Pagado"]))

print( "Precisión:", clf2.score(X_test, y_test) )


Reporte de métricas del clasificador: 
               precision    recall  f1-score   support

   No Pagado       0.21      0.64      0.32      1220
      Pagado       0.90      0.57      0.70      6744

    accuracy                           0.58      7964
   macro avg       0.56      0.61      0.51      7964
weighted avg       0.79      0.58      0.64      7964

Precisión: 0.5845052737317931


### Interpretación

| Clase | Precision | Recall | F1-score | Support |
|---|---|---|---|---|
| No Pagado | 0.21 | 0.64 | 0.32 | 1220 |
| Pagado | 0.90 | 0.58 | 0.70 | 6744 |

El accuracy global baja a 0.59 (antes 0.85), pero en este caso es buena señal:
- El modelo ya empieza a detectar los impagos.
- El recall de "No Pagado" sube de 0.00 → 0.64: ahora identifica el 64% de los créditos que realmente no se pagan.
- A cambio, el recall de "Pagado" baja de 1.00 → 0.58, lo cual es natural.

| Métrica | Sin balance | Balanceado | Interpretación |
|---|---|---|---|
| Accuracy | 0.85 | 0.59 | Baja, pero ya no es engañosa |
| Recall "No Pagado" | 0.00 | 0.64 | Mejora enorme: detecta impagos |
| Precision "No Pagado" | 0.00 | 0.21 | Mejora |
| Recall "Pagado" | 1.00 | 0.58 | Baja, ahora también predice "No Pagado" |
| F1 "No Pagado" | 0.00 | 0.32 | Mejor equilibrio entre precision y recall |

**Conclusiones:**
- El modelo ya distingue entre pagadores y morosos.
- Antes aprobaba todos los préstamos riesgosos; ahora reconoce 6 de cada 10 impagos potenciales.
- En un escenario bancario es preferible detectar más casos de alto riesgo, aunque se rechacen algunos buenos clientes, que aprobar todo y perder dinero.

In [14]:
matriz_balanceo = confusion_matrix(y_test, y_pred2 )
print(f'Matriz Confusion:\n', matriz_balanceo)

Matriz Confusion:
 [[ 780  440]
 [2869 3875]]


### Interpretación

|  | Predijo: No Pagado | Predijo: Pagado |
|---|---|---|
| **Real: No Pagado** | 776 (Verdaderos Negativos) | 444 (Falsos Positivos) |
| **Real: Pagado** | 2854 (Falsos Negativos) | 3890 (Verdaderos Positivos) |

- 776 créditos que no se pagaron fueron correctamente clasificados como No Pagado.
- 444 créditos que no se pagaron fueron clasificados como Pagado (el error más costoso).
- 2,854 créditos que sí se pagaron fueron clasificados como No Pagado.
- 3,890 créditos pagados fueron correctamente clasificados como Pagado.

Recall (No Pagado) = 776 / (776 + 444) ≈ 0.64

Recall (Pagado) = 3890 / (3890 + 2854) ≈ 0.58

## Visualizar Importancia de las variables

In [15]:
from sklearn.inspection import permutation_importance

result = permutation_importance(clf2, X_test, y_test, n_repeats=30, random_state=42)

importancia = pd.DataFrame({
    "Variable": X.columns,
    "Importancia Media": result.importances_mean,
    "Desviación": result.importances_std
}).sort_values("Importancia Media", ascending=False)

print(importancia)

               Variable  Importancia Media  Desviación
1              int_rate           0.040231    0.004291
6            annual_inc           0.023970    0.002708
2            grade_code           0.015934    0.002039
5   home_ownership_code           0.010309    0.002985
4       addr_state_code           0.006010    0.002215
7                   dti           0.004353    0.002645
8            revol_util           0.000343    0.000236
0           funded_amnt           0.000080    0.001289
9  pub_rec_bankruptcies           0.000071    0.000096
3          purpose_code          -0.001892    0.001047


In [16]:
import plotly.express as px
fig4 = px.bar(importancia, x="Variable", y="Importancia Media",
              error_y="Desviación", title="Importancia de características (Regresión Logística - Permutation Importance)",
              text_auto=".3f", color="Variable")
fig4.update_layout(width=800, height=600)
fig4.show()

### Interpretación
Las variables más importantes son **int_rate** (0.040), **annual_inc** (0.025) y **grade_code** (0.016), seguidas de home_ownership_code, addr_state_code y dti. Las demás tienen una importancia prácticamente nula.

## Normalización de datos
Muchos algoritmos (regresión logística, SVM, KNN, redes neuronales) son sensibles a la escala de los datos. Si una variable tiene un rango mucho mayor que otra, puede dominar el modelo. Por ejemplo:
- Edad: 18 - 70
- Ingresos: 10,000 - 100,000

**Sin escalar, el modelo podría darle más importancia a los ingresos solo por su magnitud.**

In [17]:
from sklearn.preprocessing import StandardScaler

X = prestamos_df[['funded_amnt', "int_rate", "grade_code", 'purpose_code', 'addr_state_code',
                  'home_ownership_code', 'annual_inc', 'dti', 'revol_util',
                  'pub_rec_bankruptcies']]

y = prestamos_df["repaid"]

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.4, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

RL_scaled = LogisticRegression(random_state=0, class_weight='balanced')
RL_scaled.fit(X_train_scaled, y_train)

y_pred_scaled = RL_scaled.predict(X_test_scaled)

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_scaled, target_names=["No Pagado", "Pagado"]))


Reporte de clasificación:
              precision    recall  f1-score   support

   No Pagado       0.22      0.61      0.32      1177
      Pagado       0.90      0.62      0.73      6787

    accuracy                           0.61      7964
   macro avg       0.56      0.61      0.53      7964
weighted avg       0.80      0.61      0.67      7964



In [18]:
print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, y_pred_scaled))


Matriz de Confusión:
[[ 719  458]
 [2611 4176]]


### Comparación de resultados sin normalizar VS con normalización

| Configuración | Verdaderos "No Pagado" | Falsos "Pagado" | Falsos "No Pagado" | Verdaderos "Pagado" | Accuracy |
|---|---|---|---|---|---|
| Balanceado sin normalizar | 776 | 444 | 2854 | 3890 | 0.59 |
| Balanceado con normalización | 719 | 458 | 2611 | 4176 | 0.61 |

**¿Por qué no mejora mucho con normalización?**
La regresión logística es un modelo lineal. Se beneficia de la normalización en estabilidad numérica y convergencia (ya no aparece el warning), pero su frontera de decisión no cambia drásticamente. El accuracy sube ligeramente de 0.59 a 0.61.

Como la regresión logística entrega probabilidades, otra opción es mover el umbral de decisión.

## Importancia de las variables después del escalado

In [19]:
from sklearn.inspection import permutation_importance

result = permutation_importance(RL_scaled, X_test_scaled, y_test, n_repeats=30, random_state=42)

importancia = pd.DataFrame({
    "Variable": X.columns,
    "Importancia Media": result.importances_mean,
    "Desviación": result.importances_std
}).sort_values("Importancia Media", ascending=False)

print(importancia)

               Variable  Importancia Media  Desviación
1              int_rate           0.039101    0.004709
6            annual_inc           0.009141    0.002740
5   home_ownership_code           0.002666    0.001302
3          purpose_code           0.001825    0.002221
4       addr_state_code           0.000013    0.002108
2            grade_code          -0.000017    0.000158
7                   dti          -0.000155    0.000799
0           funded_amnt          -0.001268    0.002061
9  pub_rec_bankruptcies          -0.001863    0.000889
8            revol_util          -0.002147    0.001014


In [20]:
import plotly.express as px
fig4 = px.bar(importancia, x="Variable", y="Importancia Media",
              error_y="Desviación", title="Importancia de características (Regresión Logística - Permutation Importance)",
              text_auto=".3f", color="Variable")
fig4.update_layout(width=800, height=600)
fig4.show()

### Interpretación
Después del escalado, **int_rate** (0.039) sigue siendo por mucho la variable más importante, seguida de **annual_inc** (0.009). El resto queda prácticamente en cero, lo que indica que la tasa de interés concentra casi toda la información para predecir el impago.

## Ajustando el umbral de decisión
- El umbral de decisión (threshold) es el valor a partir del cual una probabilidad se convierte en clase positiva (1) o negativa (0).
- Por defecto el umbral es 0.5: si la probabilidad es ≥ 0.5 se predice clase 1; si es menor, clase 0.
- En este caso la probabilidad corresponde a "Pagado". **Subir el umbral** hace que el modelo sea más exigente para decir "Pagado", así que detecta más "No Pagado". **Bajarlo** hace lo contrario.

Para elegir el mejor umbral se pueden usar la matriz de confusión, el F1-score o la curva ROC y AUC.

In [21]:
y_prob = RL_scaled.predict_proba(X_test_scaled)[:, 1]

umbral = 0.45
y_pred_umbral = (y_prob >= umbral).astype(int)

print(f"\nReporte de clasificación con umbral = {umbral}")
print(classification_report(y_test, y_pred_umbral, target_names=["No Pagado", "Pagado"]))

print("\nMatriz de Confusión con umbral de decisión a 0.45:")
print(confusion_matrix(y_test, y_pred_umbral))


Reporte de clasificación con umbral = 0.45
              precision    recall  f1-score   support

   No Pagado       0.24      0.50      0.32      1177
      Pagado       0.89      0.72      0.80      6787

    accuracy                           0.69      7964
   macro avg       0.57      0.61      0.56      7964
weighted avg       0.80      0.69      0.73      7964


Matriz de Confusión con umbral de decisión a 0.45:
[[ 588  589]
 [1881 4906]]


In [22]:
y_prob = RL_scaled.predict_proba(X_test_scaled)[:, 1]

umbral = 0.40
y_pred_umbral = (y_prob >= umbral).astype(int)

print(f"\nReporte de clasificación con umbral = {umbral}")
print(classification_report(y_test, y_pred_umbral, target_names=["No Pagado", "Pagado"]))

print("\nMatriz de Confusión con umbral de decisión a 0.40:")
print(confusion_matrix(y_test, y_pred_umbral))


Reporte de clasificación con umbral = 0.4
              precision    recall  f1-score   support

   No Pagado       0.26      0.37      0.30      1177
      Pagado       0.88      0.82      0.85      6787

    accuracy                           0.75      7964
   macro avg       0.57      0.59      0.58      7964
weighted avg       0.79      0.75      0.77      7964


Matriz de Confusión con umbral de decisión a 0.40:
[[ 433  744]
 [1250 5537]]


In [23]:
y_prob = RL_scaled.predict_proba(X_test_scaled)[:, 1]

umbral = 0.55
y_pred_umbral = (y_prob >= umbral).astype(int)

print(f"\nReporte de clasificación con umbral = {umbral}")
print(classification_report(y_test, y_pred_umbral, target_names=["No Pagado", "Pagado"]))

print("\nMatriz de Confusión con umbral de decisión a 0.55")
print(confusion_matrix(y_test, y_pred_umbral))


Reporte de clasificación con umbral = 0.55
              precision    recall  f1-score   support

   No Pagado       0.21      0.76      0.33      1177
      Pagado       0.92      0.50      0.65      6787

    accuracy                           0.54      7964
   macro avg       0.57      0.63      0.49      7964
weighted avg       0.82      0.54      0.60      7964


Matriz de Confusión con umbral de decisión a 0.55
[[ 893  284]
 [3387 3400]]


In [24]:
y_prob = RL_scaled.predict_proba(X_test_scaled)[:, 1]

umbral = 0.60
y_pred_umbral = (y_prob >= umbral).astype(int)

print(f"\nReporte de clasificación con umbral = {umbral}")
print(classification_report(y_test, y_pred_umbral, target_names=["No Pagado", "Pagado"]))

print("\nMatriz de Confusión con umbral de decisión a 0.60")
print(confusion_matrix(y_test, y_pred_umbral))


Reporte de clasificación con umbral = 0.6
              precision    recall  f1-score   support

   No Pagado       0.19      0.84      0.31      1177
      Pagado       0.93      0.38      0.54      6787

    accuracy                           0.45      7964
   macro avg       0.56      0.61      0.43      7964
weighted avg       0.82      0.45      0.51      7964


Matriz de Confusión con umbral de decisión a 0.60
[[ 989  188]
 [4206 2581]]


## Interpretación de resultados

| Umbral | Verdaderos No Pagados | Verdaderos Pagados | Recall "No Pagado" | Accuracy | Observación |
|---|---|---|---|---|---|
| 0.60 | 989 | 2581 | 0.84 | 0.45 | Detecta casi todos los impagos, pero rechaza muchos buenos clientes |
| 0.55 | 890 | 3400 | 0.76 | 0.54 | Aumenta la detección de impagos |
| 0.50 | 719 | 4176 | 0.61 | 0.61 | Umbral por defecto |
| 0.45 | 588 | 4906 | 0.50 | 0.69 | Buen equilibrio general |
| 0.40 | 433 | 5537 | 0.37 | 0.75 | Detecta menos impagos |

**Conclusiones:**
- Al bajar el umbral (0.40), el modelo predice más "Pagado": mejora los aciertos de pagos, pero pierde sensibilidad a los impagos.
- Al subir el umbral (0.60), el modelo es más exigente para decir "Pagado": detecta más impagos, pero marca como "No Pagado" muchos préstamos que sí se pagaron.
- Un umbral entre 0.45 y 0.50 logra un buen equilibrio entre detectar un número razonable de impagos y no sacrificar demasiado la precisión global.

## Implementación del modelo para Predicciones sobre nuevos valores (ficticios)

In [25]:
yhat_predmanual = clf2.predict( [[5000, 15.5 , 5, 3, 4, 4, 30000, 12, 12, 0 ]] )
yhat_predmanual

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LogisticRegression was fitted with feature names



array([0])